# Spark Assignment – Retail Transactions
Business Intelligence and Data Engineering, MSc in Business Analytics

In this notebook I analyse about 1 million online retail transactions using PySpark.
Part 1 answers the Spark SQL questions and Part 2 clusters the customers with KMeans.

## Setup
First I create the Spark session. It runs locally on my laptop using all the CPU cores.
(Spark needs Java 17 or 21 installed.)

In [1]:
# !pip install pyspark   # uncomment if pyspark is not installed

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, LongType, IntegerType,
                               DoubleType, StringType, BooleanType, TimestampType)

spark = (SparkSession.builder
         .master("local[*]")
         .appName("retail-assignment")
         .config("spark.driver.memory", "4g")
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")  # less noise in the output
spark.version

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/29 23:53:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'4.0.1'

## Loading the data
I give the schema myself instead of using `inferSchema`, so Spark doesn't need an extra pass over the file
and the `Date` column is read as a timestamp. Change `DATA_PATH` if the csv is somewhere else.

In [2]:
import os, zipfile

DATA_PATH = "retail-dataset.csv"   # put the csv in the same folder as the notebook

# the dataset was given as a zip, so if only the zip is there I extract the csv from it
if not os.path.exists(DATA_PATH) and os.path.exists(DATA_PATH + ".zip"):
    with zipfile.ZipFile(DATA_PATH + ".zip") as z:
        csv_name = [n for n in z.namelist() if n.endswith(".csv") and "__MACOSX" not in n][0]
        DATA_PATH = z.extract(csv_name, ".")
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError("Put retail-dataset.csv (or the .zip) next to this notebook or change DATA_PATH")

schema = StructType([
    StructField("Transaction_ID", LongType()),
    StructField("Date", TimestampType()),
    StructField("Customer_Name", StringType()),
    StructField("Product", StringType()), # it's a list but stored as text
    StructField("Total_Items", IntegerType()),
    StructField("Total_Cost", DoubleType()),
    StructField("Payment_Method", StringType()),
    StructField("City", StringType()),
    StructField("Store_Type", StringType()),
    StructField("Discount_Applied", BooleanType()),
    StructField("Customer_Category", StringType()),
    StructField("Season", StringType()),
    StructField("Promotion", StringType()),
])

retail = spark.read.csv(DATA_PATH, header=True, schema=schema)
retail.printSchema()

root
 |-- Transaction_ID: long (nullable = true)
 |-- Date: timestamp (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Total_Items: integer (nullable = true)
 |-- Total_Cost: double (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Store_Type: string (nullable = true)
 |-- Discount_Applied: boolean (nullable = true)
 |-- Customer_Category: string (nullable = true)
 |-- Season: string (nullable = true)
 |-- Promotion: string (nullable = true)



I cache the dataframe because I will query it many times below.

In [ ]:
retail.cache()
print("rows:", retail.count())
retail.show(5)